# Tokenization Validation

Trace one document from normalized IR through the production sequence projection and token codec.
The notebook is meant for debugging ordering, schema-version changes, and unknown-token behavior
without reimplementing tokenization logic outside `src/motifml/`.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

from motifml.datasets.motif_ir_corpus_dataset import MotifIrDocumentRecord
from motifml.ir.serialization import deserialize_document
from motifml.pipelines.feature_extraction.nodes import extract_features
from motifml.pipelines.normalization.models import NormalizedIrVersionMetadata
from motifml.training.model_input import TokenizedDocumentRow
from motifml.training.sequence_schema import coerce_sequence_schema_contract
from motifml.training.special_token_policy import coerce_special_token_policy
from motifml.training.token_codec import (
    coerce_frozen_vocabulary,
    decode_token_ids_to_strings,
    encode_projected_events_to_tokens,
    encode_token_strings_to_ids,
)
from motifml.training.token_ordering import (
    expand_sequence_event_spans,
    validate_sequence_event_order,
)

pd.set_option("display.max_colwidth", 120)


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the MotifML repository root.")


repo_root = _find_repo_root(Path.cwd())
fixture_root = Path(
    os.environ.get(
        "MOTIFML_TRAINING_FIXTURE_ROOT",
        repo_root / "tests" / "fixtures" / "training",
    )
).resolve()
golden_ir_root = Path(
    os.environ.get(
        "MOTIFML_GOLDEN_IR_ROOT",
        repo_root / "tests" / "fixtures" / "ir" / "golden",
    )
).resolve()
document_name = os.environ.get(
    "MOTIFML_TOKENIZATION_DOCUMENT",
    "ensemble_polyphony_controls",
).strip()
relative_path = f"{document_name}.json"
document_path = golden_ir_root / f"{document_name}.ir.json"
row_path = next(
    path
    for path in sorted(
        (fixture_root / "representative_rows").rglob(f"{relative_path}.row.json")
    )
    if path.is_file()
)


def load_json(path: Path) -> object:
    return json.loads(path.read_text(encoding="utf-8"))


parameters = yaml.safe_load(
    (repo_root / "conf" / "base" / "parameters.yml").read_text(encoding="utf-8")
)
persisted_row = TokenizedDocumentRow.from_row_dict(load_json(row_path))
document = deserialize_document(document_path.read_bytes())
sequence_schema = coerce_sequence_schema_contract(parameters["sequence_schema"])
normalized_ir_version = NormalizedIrVersionMetadata(
    normalized_ir_version=persisted_row.normalized_ir_version,
    contract_name=str(parameters["normalization"]["contract_name"]),
    contract_version=str(parameters["normalization"]["contract_version"]),
    serialized_document_format=str(
        parameters["normalization"]["serialized_document_format"]
    ),
    normalization_strategy=str(parameters["normalization"]["normalization_strategy"]),
    upstream_ir_schema_version=str(
        parameters["ir_build_metadata"]["ir_schema_version"]
    ),
    task_agnostic_guarantees=tuple(
        parameters["normalization"]["task_agnostic_guarantees"]
    ),
)
feature_set = extract_features(
    [MotifIrDocumentRecord(relative_path=relative_path, document=document)],
    normalized_ir_version,
    parameters["feature_extraction"],
    sequence_schema,
)
feature_record = feature_set.records[0]
projection = feature_record.projection
validate_sequence_event_order(projection.events, context=relative_path)
time_resolution = int(
    load_json(fixture_root / "vocabulary.json")["construction_parameters"][
        "time_resolution"
    ]
)
special_token_policy = coerce_special_token_policy(persisted_row.special_token_policy)
spans = expand_sequence_event_spans(
    projection.events,
    time_resolution=time_resolution,
    ordering_context=relative_path,
    note_payload_fields=sequence_schema.note_payload_fields,
)
token_strings = encode_projected_events_to_tokens(
    projection.events,
    time_resolution=time_resolution,
    ordering_context=relative_path,
    note_payload_fields=sequence_schema.note_payload_fields,
    special_token_policy=special_token_policy,
)
vocabulary_payload = load_json(fixture_root / "vocabulary.json")
vocabulary = coerce_frozen_vocabulary(vocabulary_payload)
token_ids = encode_token_strings_to_ids(
    token_strings,
    vocabulary=vocabulary,
    special_token_policy=special_token_policy,
)
decoded_tokens = decode_token_ids_to_strings(token_ids, vocabulary=vocabulary)
unknown_token_rows = [
    {
        "token_index": index,
        "token": token,
        "mapped_to": decoded_token,
    }
    for index, (token, decoded_token) in enumerate(
        zip(token_strings, decoded_tokens, strict=True)
    )
    if decoded_token == "<unk>" and token != "<unk>"
]
persisted_row_match = tuple(token_ids) == persisted_row.token_ids

display(
    Markdown(
        "## Tokenization Overview\n\n"
        f"- Document: `{relative_path}`\n"
        f"- Sequence Schema Version: `{sequence_schema.sequence_schema_version}`\n"
        f"- Derived Feature Version: `{feature_set.parameters.feature_version}`\n"
        f"- Persisted Feature Version: `{persisted_row.feature_version}`\n"
        f"- Event Count: `{len(projection.events)}`\n"
        f"- Token Count: `{len(token_strings)}`\n"
        f"- Unknown Token Count: `{len(unknown_token_rows)}`\n"
    )
)

In [ ]:
trace_rows = [
    {
        "event_index": span.event_index,
        "event_type": span.event_type,
        "start_time": f"{span.start_time.numerator}/{span.start_time.denominator}",
        "time_shift_ticks": span.time_shift_ticks,
        "tokens": " | ".join(span.tokens),
    }
    for span in spans[:12]
]
display(
    Markdown(
        "## Tokenization Trace\n\n"
        "The table below shows the first projected events after sequence-order validation and span expansion."
    )
)
display(pd.DataFrame(trace_rows))

In [ ]:
display(
    Markdown(
        "## Unknown Token Mapping\n\n"
        f"- Unknown Token Count: `{len(unknown_token_rows)}`\n"
        f"- Vocabulary Version: `{vocabulary_payload['vocabulary_version']}`\n"
        f"- Unknown Token Id: `{vocabulary.token_to_id['<unk>']}`\n"
    )
)
display(pd.DataFrame(unknown_token_rows[:12]))

In [ ]:
comparison_frame = pd.DataFrame(
    {
        "token_index": list(range(min(20, len(token_strings)))),
        "encoded_token": list(token_strings[:20]),
        "decoded_token": list(decoded_tokens[:20]),
        "persisted_token_id": list(persisted_row.token_ids[:20]),
        "recomputed_token_id": list(token_ids[:20]),
    }
)
display(
    Markdown(
        "## Persisted Row Comparison\n\n"
        f"- Persisted Row Match: `{persisted_row_match}`\n"
        f"- Persisted Model Input Version: `{persisted_row.model_input_version}`\n"
        f"- Persisted Vocabulary Version: `{persisted_row.vocabulary_version}`\n"
        f"- Persisted Token Count: `{persisted_row.token_count}`\n"
    )
)
display(comparison_frame)